# Phase 1 - DataLoader Smoke Test

This notebook mirrors `scripts/check_dataloader.py`. It checks the AwA2 manifest, the stable class mapping, the ResNet preprocessing pipeline, and the tensor ranges before and after ImageNet normalization.

## Imports

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from src.data import (
    AwA2Dataset,
    build_dataloaders,
    build_debug_transform,
    denormalize_batch,
    infer_class_map_path,
    load_class_mapping,
)
from src.utils import set_seed

PROJECT_ROOT

## Configuration

The debug manifest is preferred when it exists because it is faster to inspect. The full manifest is used as a fallback.

In [ ]:
SEED = 42
BATCH_SIZE = 8
NUM_WORKERS = 0

debug_manifest = PROJECT_ROOT / "data" / "AWA2" / "awa2_manifest_debug.csv"
full_manifest = PROJECT_ROOT / "data" / "AWA2" / "awa2_manifest.csv"
MANIFEST_PATH = debug_manifest if debug_manifest.exists() else full_manifest
CLASS_MAP_PATH = infer_class_map_path(MANIFEST_PATH)

set_seed(SEED)
MANIFEST_PATH, CLASS_MAP_PATH

## Manifest and class mapping checks

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
class_to_idx = load_class_mapping(CLASS_MAP_PATH)

display(manifest.head())
display(manifest["split"].value_counts())
display(pd.DataFrame(class_to_idx.items(), columns=["class_name", "label"]).head(10))

## Build DataLoaders

Training uses `shuffle=True`; validation and test use deterministic ordering.

In [ ]:
loaders = build_dataloaders(
    manifest_path=MANIFEST_PATH,
    class_map_path=CLASS_MAP_PATH,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

for split, loader in loaders.items():
    dataset = loader.dataset
    print(f"{split}: {len(dataset)} images, {len(dataset.visible_classes)} visible classes, {len(dataset.classes)} mapped classes")

## Inspect one normalized batch

In [ ]:
images, labels, class_names, paths = next(iter(loaders["train"]))

print("image batch shape:", tuple(images.shape))
print("label batch shape:", tuple(labels.shape))
print("first class names:", list(class_names[:5]))
print("first image path:", paths[0])
print("requires_grad:", images.requires_grad)

## Tensor ranges before and after normalization

The normalized batch can have values outside `[0, 1]` because ImageNet mean/std normalization shifts and rescales each channel. The denormalized batch is clamped only for inspection.

In [ ]:
def tensor_stats(name, tensor):
    print(
        f"{name}: min={tensor.min().item():.4f} "
        f"max={tensor.max().item():.4f} "
        f"mean={tensor.mean().item():.4f} "
        f"std={tensor.std().item():.4f}"
    )


tensor_stats("normalized", images)

with torch.no_grad():
    denorm = denormalize_batch(images).clamp(0.0, 1.0)

tensor_stats("denormalized", denorm)

## Pre-normalization sample

This uses the same resize/crop/to-tensor pipeline but intentionally skips ImageNet normalization.

In [ ]:
debug_dataset = AwA2Dataset(
    manifest_path=MANIFEST_PATH,
    split="train",
    transform=build_debug_transform(),
    class_map_path=CLASS_MAP_PATH,
)

raw_tensor, raw_label, raw_class, raw_path = debug_dataset[0]
print("shape:", tuple(raw_tensor.shape))
print("label:", raw_label)
print("class:", raw_class)
print("path:", raw_path)
tensor_stats("pre-normalization", raw_tensor)